# Topic 23 — N-grams
### Theory → from-scratch n-grams → CountVectorizer(ngram_range) → why n-grams help toxicity detection.

Bag of Words (Topic 22) treats every word independently, throwing away order — "not good" and
"good not" look identical. **N-grams** are contiguous sequences of `n` tokens, which recover a bit
of local word order and context.

- **Unigram** (n=1): single words — `["you", "are", "stupid"]`
- **Bigram** (n=2): pairs of consecutive words — `["you are", "are stupid"]`
- **Trigram** (n=3): triples — `["you are stupid"]`
- **Character n-grams**: same idea, but over characters instead of words — useful for catching
  misspellings/obfuscation like "st*pid" or "s t u p i d".

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

## 1. From-scratch n-gram generation

In [ ]:
def generate_ngrams(text, n):
    tokens = text.lower().split()
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

sentence = "you are stupid"

print("unigrams:", generate_ngrams(sentence, 1))
print("bigrams: ", generate_ngrams(sentence, 2))
print("trigrams:", generate_ngrams(sentence, 3))

# The key insight: negation changes meaning, and bigrams start to capture that
print()
print("bigrams of 'not good':", generate_ngrams("not good", 2))
print("bigrams of 'good not':", generate_ngrams("good not", 2))
# As unigrams (Topic 22), both sentences produce the identical bag {"good", "not"}.
# As bigrams, they produce DIFFERENT features -- some of the order information is preserved.

## 2. `CountVectorizer` / `TfidfVectorizer` with `ngram_range`

`ngram_range=(min_n, max_n)` tells sklearn to generate ALL n-grams in that range, not just one size —
e.g. `(1, 2)` includes both unigrams AND bigrams together.

In [ ]:
docs = [
    "you are stupid",
    "you are not stupid",
    "great job today",
]

vec_unigram = CountVectorizer(ngram_range=(1, 1))
X_uni = vec_unigram.fit_transform(docs)
print("unigram vocabulary:", vec_unigram.get_feature_names_out())

vec_bigram = CountVectorizer(ngram_range=(2, 2))
X_bi = vec_bigram.fit_transform(docs)
print("\nbigram vocabulary:", vec_bigram.get_feature_names_out())

vec_both = CountVectorizer(ngram_range=(1, 2))
X_both = vec_both.fit_transform(docs)
print("\nunigram+bigram vocabulary:", vec_both.get_feature_names_out())
print("shape:", X_both.shape, "(more features than unigrams alone)")

In [ ]:
print(pd.DataFrame(X_both.toarray(), columns=vec_both.get_feature_names_out(), index=docs))
# Notice: "you are stupid" and "you are not stupid" now look meaningfully different --
# the bigram "not stupid" only appears in the second sentence.

## 3. Character n-grams — catching obfuscated toxic text

Word-level n-grams can be dodged by deliberate misspelling: "st*pid", "stooopid", "s-t-u-p-i-d".
Character n-grams are more robust to this because they look at chunks of LETTERS, not whole words.

In [ ]:
vec_char = CountVectorizer(analyzer="char", ngram_range=(3, 3))
obfuscated_docs = ["you are stupid", "you are st*pid", "you are stooopid"]

X_char = vec_char.fit_transform(obfuscated_docs)
char_df = pd.DataFrame(X_char.toarray(), columns=vec_char.get_feature_names_out(), index=obfuscated_docs)

# Show just a few informative columns for readability
relevant_cols = [c for c in char_df.columns if "stu" in c or "stu" in c or "oop" in c or "tup" in c]
print(char_df[[c for c in char_df.columns if "tup" in c or "oop" in c]] if any("tup" in c or "oop" in c for c in char_df.columns) else char_df.iloc[:, :10])
print("\nfull vocabulary size (character trigrams):", len(vec_char.get_feature_names_out()))
# Even obfuscated spellings share SOME overlapping character trigrams with the original word,
# giving the model partial signal that word-level n-grams would completely miss.

## 4. The tradeoff: n-grams massively increase vocabulary size

More `n`, or combining multiple n's, means a much bigger (and sparser) feature space — more
potential signal, but more computation, more risk of overfitting on rare n-grams, and more noise.

In [ ]:
bigger_docs = [
    "you are stupid and worthless", "i hate you so much", "get lost loser",
    "nobody wants you here", "great job today team", "have a wonderful day",
]

for ngram_range in [(1,1), (1,2), (1,3)]:
    v = CountVectorizer(ngram_range=ngram_range)
    X = v.fit_transform(bigger_docs)
    print(f"ngram_range={ngram_range}: vocabulary size = {X.shape[1]}")
# Vocabulary size grows quickly as you widen ngram_range -- min_df/max_features (Topic 22)
# become more important to control this.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Generate trigrams from-scratch for 3 of your own sentences using generate_ngrams().
# 2. Build a TfidfVectorizer(ngram_range=(1,2)) on bigger_docs and inspect which bigrams get
#    the highest average TF-IDF weight (preview of Topic 24).
# 3. Try char-level ngram_range=(2,4) on a couple of deliberately misspelled toxic-sounding words
#    (make them up, don't need real data) and see how many overlapping trigrams they share with
#    the correctly-spelled version.
# 4. For your cyberbullying paper: would you use ngram_range=(1,1), (1,2), or (1,3) as your default,
#    given the tradeoff between capturing phrases and vocabulary explosion? Justify briefly.

---
### Next up: **Topic 24 — TF-IDF** — one of your strongest NLP concepts to nail down.

Say "next" when you're ready.